# 02 — Preprocessing

For each video in the locked eval set:
1. Extract the audio track to `data/audio/{vid}.wav` (16 kHz mono PCM, Whisper-compatible).
2. Strip audio to make a silent video at `data/silent_video/{vid}_silent.mp4` (used in S2 visual-only).
3. Run Whisper-small for ASR transcript at `data/transcripts/{vid}.txt`.
4. Build the mismatched-transcript pool (used in S5 lexical-override stage).

**Runtime:** ~30–45 minutes for 120 videos on H100 (Whisper is the bottleneck).

**Safe to re-run:** every step skips files that already exist.


In [ ]:
import os, sys, json, subprocess
REPO = '/content/omnimodel-research'
if os.path.exists(REPO):
    %cd $REPO
    sys.path.insert(0, REPO)

from pathlib import Path

with open('data/eval_samples.json') as f:
    samples = json.load(f)
print(f'Preprocessing {len(samples)} videos.')

VIDEO_DIR = 'data/videos'
AUDIO_DIR = 'data/audio'
SILENT_DIR = 'data/silent_video'
TRANSCRIPT_DIR = 'data/transcripts'
TRANSCRIPT_MISMATCHED_DIR = 'data/transcripts_mismatched'

for d in [AUDIO_DIR, SILENT_DIR, TRANSCRIPT_DIR, TRANSCRIPT_MISMATCHED_DIR]:
    os.makedirs(d, exist_ok=True)


## 1. Audio extraction (ffmpeg)

In [ ]:
for s in samples:
    vid = s['video_id']
    in_path = f'{VIDEO_DIR}/{vid}.mp4'
    out_path = f'{AUDIO_DIR}/{vid}.wav'
    if os.path.exists(out_path):
        continue
    if not os.path.exists(in_path):
        print(f'  Skipping {vid}: no source video')
        continue
    subprocess.run([
        'ffmpeg', '-y', '-i', in_path,
        '-vn', '-acodec', 'pcm_s16le',
        '-ar', '16000', '-ac', '1',
        out_path,
    ], capture_output=True)
print(f'Audio files: {len([f for f in os.listdir(AUDIO_DIR) if f.endswith(".wav")])}')


## 2. Silent video creation (ffmpeg, audio-stripped)

In [ ]:
for s in samples:
    vid = s['video_id']
    in_path = f'{VIDEO_DIR}/{vid}.mp4'
    out_path = f'{SILENT_DIR}/{vid}_silent.mp4'
    if os.path.exists(out_path):
        continue
    if not os.path.exists(in_path):
        continue
    subprocess.run([
        'ffmpeg', '-y', '-i', in_path,
        '-an', '-c:v', 'copy',
        out_path,
    ], capture_output=True)
print(f'Silent videos: {len([f for f in os.listdir(SILENT_DIR) if f.endswith(".mp4")])}')


## 3. Whisper transcripts (matched, for S4)

In [ ]:
from src.transcript_utils import transcribe_with_whisper

# Whisper-small balances speed and quality. The transcript is intentionally imperfect —
# real-world transcripts have errors, and a perfect transcript would inflate the
# text signal artificially.

for i, s in enumerate(samples):
    vid = s['video_id']
    audio_path = f'{AUDIO_DIR}/{vid}.wav'
    if not os.path.exists(audio_path):
        continue
    text = transcribe_with_whisper(audio_path, output_dir=TRANSCRIPT_DIR, model_size='small')
    if (i + 1) % 10 == 0:
        print(f'  [{i+1}/{len(samples)}] transcripts done')

ts_files = [f for f in os.listdir(TRANSCRIPT_DIR) if f.endswith('.txt')]
print(f'Transcripts: {len(ts_files)}')


## 4. Mismatched-transcript pool (for S5)

For each sample, pair it with the transcript of a DIFFERENT same-task video. This is the lexical-override condition: the audio in the played video tells the truth, but the transcript contradicts it.

We persist the pairing to disk so S5 reads it deterministically.


In [ ]:
from src.transcript_utils import build_mismatched_pairs, load_transcript

pairs = build_mismatched_pairs(samples, seed=42)
# pairs maps {qa_id: donor_video_id}

n_written = 0
for qa_id, donor_vid in pairs.items():
    out_path = f'{TRANSCRIPT_MISMATCHED_DIR}/{qa_id}.txt'
    if os.path.exists(out_path):
        n_written += 1
        continue
    donor_text = load_transcript(f'{TRANSCRIPT_DIR}/{donor_vid}.txt')
    if not donor_text:
        continue
    with open(out_path, 'w') as f:
        f.write(donor_text)
    n_written += 1

with open('data/mismatched_pairs.json', 'w') as f:
    # JSON keys must be strings — convert qa_id back when reading
    json.dump({str(k): v for k, v in pairs.items()}, f, indent=2)

print(f'Mismatched transcripts written: {n_written} / {len(pairs)}')


## Sanity check

Confirm everything we need exists for each sample. If a sample is missing any artifact, drop it from the eval set — running stages on a partial sample creates messy paired-stage analysis later.


In [ ]:
missing = []
for s in samples:
    vid = s['video_id']
    qa_id = s['qa_id']
    needed = [
        f'{VIDEO_DIR}/{vid}.mp4',
        f'{AUDIO_DIR}/{vid}.wav',
        f'{SILENT_DIR}/{vid}_silent.mp4',
        f'{TRANSCRIPT_DIR}/{vid}.txt',
        f'{TRANSCRIPT_MISMATCHED_DIR}/{qa_id}.txt',
    ]
    missing_files = [p for p in needed if not os.path.exists(p)]
    if missing_files:
        missing.append({'qa_id': qa_id, 'video_id': vid, 'missing': missing_files})

print(f'QA pairs with all artifacts: {len(samples) - len(missing)} / {len(samples)}')
if missing:
    print('Samples missing artifacts (will be dropped):')
    for m in missing[:10]:
        print(f"  qa_id={m['qa_id']} (video={m['video_id']}): missing {len(m['missing'])} files")

bad_qa_ids = {m['qa_id'] for m in missing}
clean_samples = [s for s in samples if s['qa_id'] not in bad_qa_ids]
with open('data/eval_samples_clean.json', 'w') as f:
    json.dump(clean_samples, f, indent=2)
print(f'Final clean eval set: {len(clean_samples)} samples → data/eval_samples_clean.json')


**Done.** Proceed to `03_pilot.ipynb`.